<a href="https://colab.research.google.com/github/AbdAlRahman-Odeh-99/Two_Phases_Simulation/blob/main/gmm_afa_supervised_bandit_feedback.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Multiclass GMM Supervised Simulation

In [1]:
# Install numba for JIT compilation
!pip install numba

In [2]:
# mydataset.py content (generate_data function)
import numpy as np
from sklearn.datasets import make_blobs

def generate_data(nsamples=1000,nclasses=2,nviews=10,seed=42,snrdb=9):
    sval = 10**(snrdb/20)
    rng = np.random.default_rng(seed=seed)
    mm = rng.random(size=(nclasses,nviews)) * sval # symmetric
    # shared variances
    # FIXME: currently simplify to unit variances, only means differ
    stds = 1.0 # simplification
    #mm = np.concatenate([-mm, mm],axis=0) #
    data = make_blobs(nsamples,n_features=nviews,centers=mm,cluster_std=stds,random_state=seed)
    return data[0], mm, data[1] #(observations, means, labels)

In [ ]:
# mydataset.py
import numpy as np
from sklearn.datasets import make_blobs

def generate_data(nsamples=1000,nclasses=2,nviews=10,seed=42,snrdb=9):
    sval = 10**(snrdb/20)
    rng = np.random.default_rng(seed=seed)
    mm = rng.random(size=(nclasses,nviews)) * sval # symmetric
    # shared variances
    # FIXME: currently simplify to unit variances, only means differ
    stds = 1.0 # simplification
    #mm = np.concatenate([-mm, mm],axis=0) #
    data = make_blobs(nsamples,n_features=nviews,centers=mm,cluster_std=stds,random_state=seed)
    return data[0], mm, data[1] #(observations, means, labels)


In [3]:
import os
import sys
import time
import pandas as pd
import numpy as np
from numba import jit
# from mydataset import generate_data # generate_data is defined in a previous cell


# --- Replaced argparse with direct variable assignments for notebook execution ---
# parser = argparse.ArgumentParser()
# parser.add_argument("--seed",type=int,default=42,help="random seed for simulation")
# parser.add_argument("--num_views",type=int,default=20,help="number of views")
# parser.add_argument("--num_classes",type=int,default=4,help="number of classes")
# parser.add_argument("--num_rounds",type=int,default=1000,help="number of simulations")
# parser.add_argument("--num_trials",type=int,default=20,help="number of trials")
# parser.add_argument("--budget_ratio",type=int,default=0.7,help="budget ratio")
# parser.add_argument("--output",type=str,default="output_multiclass_gmm_supervised.csv",
#                     help="output file name")

# Define simulation parameters directly for Colab execution
class Args:
    def __init__(self):
        self.seed = 42
        self.num_views = 20
        self.num_classes = 4
        self.num_rounds = 1000
        self.num_trials = 20
        self.budget_ratio = 0.7
        self.output = "output_multiclass_gmm_supervised.csv"

args = Args()
# -----------------------------------------------------------------------------

@jit
def multiclass_risk(diff_mean_sq_mat):
    nc = diff_mean_sq_mat.shape[1]
    err_rate = bhattacharyya_error_rate(diff_mean_sq_mat) # (nc, nc) error rates NOTE: diagonal are zeros
    denom = 1.0/nc/(nc-1)
    return 0.5 * denom * np.sum(err_rate) # only half of the total sum as we want off-diagonal terms

@jit
def bhattacharyya_error_rate(diff_mean_sq_mat):
    d_norm = np.sum(diff_mean_sq_mat,axis=0) # (nc, nc)
    acc = np.exp(-0.125 * d_norm) # 1/8 * |\delta\mu|^2
    return np.maximum(1.0 - acc,0.0)

@jit
def pred_linear_cla(x_observe,class_means):
    mean_tr = class_means.T # (v,nc)
    diff_mean_sq = mean_tr[:,:,None] - mean_tr[:,None,:] # (sel, nc, nc)
    pairwise_mean_avg = 0.5 * (mean_tr[:,:,None] + mean_tr[:,None,:])
    inner_prod = np.sum(
        (x_observe[:,None,None] - pairwise_mean_avg) * diff_mean_sq,
        axis=0) > 0 # bool(nc,nc)
    # mask the diagonal
    np.fill_diagonal(inner_prod,False)
    # prediction
    y_pred = np.argmax(np.sum(inner_prod,axis=1))
    return y_pred

def greedy_oracle(diff_mean_sq,costs, omd_lambda, remain_budget, free_indices):
    """
        diff_mean_sq: shape (nview, nc, nc)
    """
    # Bhattacharyya distance
    # err probability upper bound
    nview = len(costs)
    current_cost = 0.0
    current_objective = 0.0
    sel_set = [] # a copy
    avail_elements = set(range(nview))
    # harvest from free views first
    for i in list(free_indices):
        tmp_select = sel_set + [i]
        margin_gain = multiclass_risk(diff_mean_sq[np.array(tmp_select)]) - current_objective
        if margin_gain > 0:
            sel_set.append(i)
            current_objective += margin_gain
            avail_elements.remove(i)
    copy_set = list(sel_set)
    while avail_elements and current_cost < remain_budget:
        best_margin = -1
        best_add = None
        for i in avail_elements:
            cost_i = costs[i]
            if current_cost + cost_i <= remain_budget:
                tmp_select = sel_set + [i]
                margin_gain = multiclass_risk(diff_mean_sq[np.array(tmp_select)]) - current_objective
                # now there is no zero cost elements, add epsilon for safety
                gain_ratio = margin_gain / (cost_i + 1e-9)
                # only add it if the margin is greater than the shadow price
                if gain_ratio > omd_lambda and gain_ratio > best_margin:
                    best_margin = gain_ratio
                    best_add = i
        if best_add is not None:
            sel_set.append(best_add)
            current_cost += costs[best_add]
            current_objective = multiclass_risk(diff_mean_sq[np.array(sel_set)])
            avail_elements.remove(best_add)
        else:
            break
    # Giant item check
    greedy_solution_reward = current_objective
    greedy_solution_indices = list(sel_set)
    best_single_item = None
    best_giant_reward = -np.inf
    for i in range(nview):
        if i in copy_set:
            continue
        cost_i = costs[i]
        if 0 < cost_i <= remain_budget:
            tmp_giant_indices = copy_set + [i]
            reward_with_giant = multiclass_risk(diff_mean_sq[np.array(tmp_giant_indices)])
            if reward_with_giant > best_giant_reward:
                best_giant_reward = reward_with_giant
                best_single_item = i

    # compare against the giant item
    if best_single_item is not None and best_giant_reward > greedy_solution_reward:
        final_indices = free_indices + [best_single_item]
    else:
        final_indices = greedy_solution_indices
    # convert to boolean vector
    return np.array([idx in final_indices for idx in range(nview)]).astype("bool")


def simulation(xdata,costs,num_rounds,budget_ratio,means,y_labels,seed)-> dict:
    rng = np.random.default_rng(seed=seed)
    nclasses = means.shape[0] # (nc, xdim)
    nviews = means.shape[1]
    # init weights
    weights = rng.normal(size=(nclasses,nviews))
    ncounts = np.ones((nclasses,nviews)) # NOTE: Reward=1, single update, Reward=0, all updates
    one_vec = np.ones(nclasses)
    # optimization
    free_indices = np.array([idx for idx in range(nviews) if costs[idx] == 0])
    omd_lambda = 0
    lambda_max = 10.0
    step_size= 0.2
    remain_budget = num_rounds * budget_ratio # initialization
    alpha_ucb = 2.0
    lr = 1e-2
    # recording
    record_acc = np.zeros(num_rounds)

    for t in range(num_rounds):
        if t < nclasses:
            subset = np.ones(nviews).astype("bool")
            is_init = True
        else:
            # optimistic estimate
            w_diff = weights.T[:,:,None] - weights.T[:,None,:] #(views, nc, nc)
            diff_mean_sq = np.square(w_diff)
            #FIXME: add the ucb bonus on square difference...
            inv_sqrt_cnt = np.sqrt(1./ ncounts).T # (nv, nc)
            bonus_mat = inv_sqrt_cnt[:,:,None] + inv_sqrt_cnt[:,None,:]
            bonus_mat *= np.sqrt(alpha_ucb * np.log(t+1))
            optimistic_diff_mean_sq = diff_mean_sq + bonus_mat
            subset = greedy_oracle(optimistic_diff_mean_sq,costs,omd_lambda,remain_budget,free_indices)
            is_init = False

        # checking the budget
        inst_cost = np.sum(costs[subset])
        if remain_budget >= inst_cost:
            remain_budget -= inst_cost
        else:
            # override with free indices
            subset = np.zeros(nviews).astype("bool")
            subset[free_indices] = True

        # observe
        #subset = np.ones(nviews).astype("bool") # debug
        x_obs = xdata[t,subset]
        w_sub = weights[:,subset]
        if not is_init:
            # prediction
            #y_pred = np.argmax(corr)
            y_pred = pred_linear_cla(x_obs,w_sub)
        else:
            y_pred = rng.integers(nclasses)

        # for reward and update (external)
        y_true = y_labels[t]
        reward = y_pred == y_true

        # update weights
        ncounts[y_pred, subset] += 1
        if reward:
            # correct prediction
            # Only update the weight vector for the true class
            weights[y_pred, subset] += (1/ncounts[y_pred,subset]) * (x_obs - weights[y_pred,subset])
            #ncounts[y_true, subset] += 1
            #grad = -2 * (x_obs - w_sub[y_pred])
            #weights[y_pred, subset] -= lr * grad
        else:
            # incorrect prediction (complementary label update)
            # The model only knows its prediction was wrong.
            eliminated = y_pred
            elim = np.zeros(nclasses)
            elim[eliminated] = 1.0
            l_grad = -2 * (x_obs[None,:] - w_sub) # (nc, ns)
            grad =  l_grad * ( one_vec - (nclasses -1 ) * elim )[:,None]
            weights[:,subset] -= lr * grad
        # update the omd
        raw_lambda = omd_lambda + step_size * (inst_cost - budget_ratio)
        omd_lambda = max(0, min(lambda_max, raw_lambda))
        # record results
        record_acc[t] = reward
    # collecting results
    spending = budget_ratio * num_rounds - remain_budget
    reward = np.sum(record_acc)
    return {"reward":reward,"spending":spending,"avg_acc":np.mean(record_acc),"record_acc":record_acc}

def main(args):
    rng = np.random.default_rng(seed=args.seed)
    # recording
    result_pd = {"trial":[], "reward":[], "spending":[], "avg_acc":[]}
    # simulation
    time_start = time.perf_counter()
    for t in range(args.num_trials):

        x_data, means, y_labels = generate_data(args.num_rounds,
                                                args.num_classes,
                                                args.num_views,
                                                args.seed,
                                                snrdb=9)
        cost_vec = rng.random(size=(args.num_views,))
        cost_vec[0] = 0.0 # adding one free view
        cost_vec /= np.sum(cost_vec)
        trial_start = time.perf_counter()
        sim_dict = simulation(x_data,cost_vec,args.num_rounds,args.budget_ratio,means,y_labels,args.seed)
        print(f"Trial {t + 1}/{args.num_trials}... (Time elapsed: {time.perf_counter() - trial_start:.3f}s)")
        # accumulating
        result_pd['trial'].append(t)
        result_pd['reward'].append(sim_dict['reward'])
        result_pd['spending'].append(sim_dict['spending'])
        result_pd['avg_acc'].append(sim_dict['avg_acc'])
    res_df = pd.DataFrame.from_dict(result_pd)
    res_df.to_csv(args.output,index=False)
    print(f"Simulation complete, total time elapsed: {time.perf_counter() - time_start:.3f}s")
# if __name__ == "__main__":
#     main(parser.parse_args()) # Call main with the defined args object


In [4]:
main(args)

Trial 1/20... (Time elapsed: 8.456s)
Trial 2/20... (Time elapsed: 1.801s)
Trial 3/20... (Time elapsed: 2.075s)
Trial 4/20... (Time elapsed: 1.350s)
Trial 5/20... (Time elapsed: 1.359s)
Trial 6/20... (Time elapsed: 1.324s)
Trial 7/20... (Time elapsed: 1.355s)
Trial 8/20... (Time elapsed: 1.323s)
Trial 9/20... (Time elapsed: 1.326s)
Trial 10/20... (Time elapsed: 1.356s)
Trial 11/20... (Time elapsed: 2.102s)
Trial 12/20... (Time elapsed: 1.782s)
Trial 13/20... (Time elapsed: 1.322s)
Trial 14/20... (Time elapsed: 1.373s)
Trial 15/20... (Time elapsed: 1.336s)
Trial 16/20... (Time elapsed: 1.318s)
Trial 17/20... (Time elapsed: 1.351s)
Trial 18/20... (Time elapsed: 3.925s)
Trial 19/20... (Time elapsed: 1.963s)
Trial 20/20... (Time elapsed: 1.353s)
Simulation complete, total time elapsed: 39.619s


In [5]:
import pandas as pd

# Load the results from the CSV file
results_df = pd.read_csv('output_multiclass_gmm_supervised.csv')

# Display the first few rows of the DataFrame
print('Simulation Results (first 5 rows):')
print(results_df.head())

# Display some summary statistics
print('\nSummary Statistics:')
print(results_df.describe())

Simulation Results (first 5 rows):
   trial  reward    spending  avg_acc
0      0   891.0  699.085882    0.891
1      1   895.0  698.165361    0.895
2      2   897.0  699.267062    0.897
3      3   900.0  699.657196    0.900
4      4   919.0  697.424066    0.919

Summary Statistics:
          trial      reward    spending    avg_acc
count  20.00000   20.000000   20.000000  20.000000
mean    9.50000  906.400000  699.102728   0.906400
std     5.91608   11.282963    0.784427   0.011283
min     0.00000  888.000000  697.424066   0.888000
25%     4.75000  896.750000  698.439545   0.896750
50%     9.50000  907.500000  699.230464   0.907500
75%    14.25000  915.500000  699.803852   0.915500
max    19.00000  925.000000  699.995761   0.925000
